# LSTM Autoencoder 재학습 (Colab GPU)

## 사용법
1. Google Drive에 `dataset_causRCA` 폴더 업로드 (dig_twin, real_op 포함)
2. 런타임 → 런타임 유형 변경 → **GPU** 선택
3. 셀을 순서대로 실행
4. 마지막 셀에서 `edge_agent_data.zip` 다운로드
5. 배포:
   - `models_lstm/*.pt`, `*_info.json` → `edge-agent/models/`
   - `processed_data_v2/*` → `anomaly_detection/processed_data_v2/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
import pickle
import zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── 설정 ──
WINDOW = 30
HIDDEN, LATENT, N_LAYERS = 64, 32, 2
EPOCHS = 50
BATCH_SIZE = 64
LR = 1e-3

data_dir = '/content/drive/MyDrive/dataset_causRCA'
out_dir = '/content/lstm_output'

SUBSYSTEM_FEATURES = {
    'coolant': [
        'CLF_A_700307','CLF_Filter_Ok','CLT_A_700310','CLT_Level_lt_Min',
        'F_A_700313','F_Filter_Ok','HP_A_700304','HP_Pump_Ok','HP_Pump_isOff',
        'LP_A_700301','LP_Pump_Ok','LP_Pump_On','LT_A_700317','LT_Level_Ok','LT_Pump_Ok',
    ],
    'hydraulics': [
        'Hyd_A_700202','Hyd_A_700203','Hyd_A_700204','Hyd_A_700205','Hyd_A_700206',
        'Hyd_A_700207','Hyd_A_700208','Hyd_Filter_Ok','Hyd_IsEnabled','Hyd_Level_Ok',
        'Hyd_Pressure','Hyd_Pump_Ok','Hyd_Pump_On','Hyd_Pump_isOff','Hyd_Temp_lt_70',
        'Hyd_Temp_lt_80','Hyd_Valve_P_Up','Lubr_On','Lubr_P_Ok',
    ],
    'probe': [
        'MPA_A_701124','MPA_A_701125','MPA_InitPos','MPA_WorkPos','MPA_toInitPos',
        'MPA_toWorkPos','MPC_Closed','MPC_close','MPC_isOpen','MPC_open','MP_Inactive',
    ],
}

SUBSYSTEM_DIR = {
    'coolant':    'exp_coolant',
    'hydraulics': 'exp_hydraulics',
    'probe':      'exp_probe',
}

DATASET_DIR = Path(data_dir)
DIG_TWIN = DATASET_DIR / 'dig_twin'
REAL_OP = DATASET_DIR / 'real_op'   # ← 추가: real_op 경로
OUT_DIR = Path(out_dir)
PROC_DIR = OUT_DIR / 'processed_data_v2'
LSTM_DIR = OUT_DIR / 'models_lstm'
PROC_DIR.mkdir(parents=True, exist_ok=True)
LSTM_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Dataset: {DATASET_DIR}')
print(f'real_op: {REAL_OP} (exists={REAL_OP.exists()})')

In [ ]:
class LSTMEncoder(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, HIDDEN, N_LAYERS, batch_first=True, dropout=0.1)
        self.fc = nn.Linear(HIDDEN, LATENT)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])


class LSTMDecoder(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.fc = nn.Linear(LATENT, HIDDEN)
        self.lstm = nn.LSTM(HIDDEN, HIDDEN, N_LAYERS, batch_first=True, dropout=0.1)
        self.out = nn.Linear(HIDDEN, n_feat)

    def forward(self, z):
        h = self.fc(z).unsqueeze(1).repeat(1, WINDOW, 1)
        o, _ = self.lstm(h)
        return torch.sigmoid(self.out(o))


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.encoder = LSTMEncoder(n_feat)
        self.decoder = LSTMDecoder(n_feat)

    def forward(self, x):
        return self.decoder(self.encoder(x))


In [ ]:
def _b2f(v):
    s = str(v).strip().lower()
    if s == 'true': return 1.0
    if s == 'false': return 0.0
    try:
        return float(s)
    except:
        return 0.0


def load_wide(csv_path, features):
    """
    Long 포맷 (time_s, node, value, type) 또는
    Wide 포맷 (time_s, feat1, feat2, ...) 모두 지원.
    → (T × N_FEAT) DataFrame 반환
    """
    df = pd.read_csv(csv_path)

    # Wide 포맷 감지: 'node' 컬럼이 없으면 이미 wide (real_op 등)
    if 'node' not in df.columns:
        result = pd.DataFrame(index=df.index)
        for name in features:
            if name in df.columns:
                result[name] = pd.to_numeric(df[name], errors='coerce').fillna(0.0)
            else:
                result[name] = 0.0
        return result

    # Long 포맷 (dig_twin) → pivot_table
    wide = df.pivot_table(index='time_s', columns='node', values='value', aggfunc='last')
    wide = wide.reindex(columns=features)
    return wide.map(_b2f).fillna(0.0)


def make_windows(arr, w=WINDOW):
    return np.stack([arr[i:i+w] for i in range(len(arr) - w + 1)]).astype(np.float32)

In [ ]:
# ── [E-1] 전체 피처 목록 수집 ──
# dig_twin (long) + real_op (wide) 모두에서 피처 수집
print('[1/4] 피처 목록 수집...')
all_nodes = set()

# dig_twin에서 수집 (long 포맷 → node 컬럼)
for sub, folder in SUBSYSTEM_DIR.items():
    sub_path = DIG_TWIN / folder
    if not sub_path.exists():
        print(f'  경고: {sub_path} 없음')
        continue
    for exp_dir in sub_path.iterdir():
        if not exp_dir.is_dir() or exp_dir.name.startswith('.'): continue
        for run_dir in exp_dir.iterdir():
            if not run_dir.is_dir() or run_dir.name.startswith('.'): continue
            for csv_f in run_dir.glob('faultDataset_*.csv'):
                try:
                    df = pd.read_csv(csv_f)
                    if 'node' in df.columns:
                        all_nodes.update(df['node'].unique())
                except Exception as e:
                    print(f'  스킵 ({csv_f.name}): {e}')

# real_op에서 수집 (wide 포맷 → 컬럼명이 피처명)
if REAL_OP.exists():
    real_csvs = sorted(REAL_OP.glob('*.csv'))
    print(f'  real_op CSV: {len(real_csvs)}개')
    for csv_f in real_csvs[:3]:  # 처음 3개만 스캔
        try:
            df = pd.read_csv(csv_f, nrows=1)
            if 'node' in df.columns:
                all_nodes.update(df['node'].unique())
            else:
                cols = [c for c in df.columns if c != 'time_s']
                all_nodes.update(cols)
        except Exception as e:
            print(f'  스킵 ({csv_f.name}): {e}')
else:
    print('  경고: real_op 폴더 없음!')

ALL_FEATURES = sorted(all_nodes)
N_FEAT = len(ALL_FEATURES)
print(f'  전체 피처 수: {N_FEAT}')

In [ ]:
# ── [E-2] scaler + meta.json 생성 ──
# dig_twin normal + real_op 전체를 정상 데이터로 사용
print('[2/4] scaler + meta 생성...')
all_normal = []

# (A) dig_twin: cause_start 이전만 정상
for sub, folder in SUBSYSTEM_DIR.items():
    sub_path = DIG_TWIN / folder
    if not sub_path.exists(): continue
    for exp_dir in sorted(sub_path.iterdir()):
        if not exp_dir.is_dir() or exp_dir.name.startswith('.'): continue
        for run_dir in sorted(exp_dir.iterdir()):
            if not run_dir.is_dir() or run_dir.name.startswith('.'): continue
            csv_files = list(run_dir.glob('faultDataset_*.csv'))
            cause_files = list(run_dir.glob('causes.json'))
            if not csv_files or not cause_files: continue
            try:
                causes = json.loads(cause_files[0].read_text())
                cause_start = causes.get('cause_start_at', 0.0)
                wide = load_wide(csv_files[0], ALL_FEATURES)
                normal = wide[wide.index < cause_start]
                if len(normal) >= 5:
                    all_normal.append(normal)
            except Exception as e:
                print(f'  dig_twin 스킵 ({run_dir.name}): {e}')

print(f'  dig_twin 정상 세그먼트: {len(all_normal)}개')

# (B) real_op: 전체가 정상 운전 데이터
real_op_count = 0
if REAL_OP.exists():
    real_csvs = sorted(REAL_OP.glob('*.csv'))
    for csv_f in real_csvs:
        try:
            wide = load_wide(csv_f, ALL_FEATURES)
            if len(wide) >= 5:
                all_normal.append(wide)
                real_op_count += 1
        except Exception as e:
            print(f'  real_op 오류 ({csv_f.name}): {e}')
    print(f'  real_op 정상 파일: {real_op_count}개')

normal_all = pd.concat(all_normal, ignore_index=True).values.astype(np.float32)
print(f'  전체 정상 행 수: {len(normal_all)}')

col_min = normal_all.min(axis=0)
col_range = normal_all.max(axis=0) - col_min
col_range[col_range == 0] = 1.0

with open(PROC_DIR / 'scaler_info.pkl', 'wb') as f:
    pickle.dump({'min': col_min, 'range': col_range}, f)
print(f'  scaler_info.pkl 저장 (shape: {col_min.shape})')

feat_idx = {n: i for i, n in enumerate(ALL_FEATURES)}
subsystem_info = {}
for sub, feats in SUBSYSTEM_FEATURES.items():
    valid = [f for f in feats if f in feat_idx]
    subsystem_info[sub] = {'features': valid, 'indices': [feat_idx[f] for f in valid]}

meta = {'feature_names': ALL_FEATURES, 'subsystem_info': subsystem_info}
with open(PROC_DIR / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('  meta.json 저장')
for sub, info in subsystem_info.items():
    print(f'    {sub}: {len(info["features"])}개 피처')

In [ ]:
# ── [E-3] LSTM Autoencoder 학습 ──
# dig_twin normal + real_op 전체로 학습
print(f'[3/4] LSTM Autoencoder 학습 (epochs={EPOCHS})...')

for subsystem in ['coolant', 'hydraulics', 'probe']:
    indices = subsystem_info[subsystem]['indices']
    n_feat = len(indices)
    print(f'\n  [{subsystem}] n_feat={n_feat}')

    wins = []

    # (A) dig_twin normal windows
    for sub, folder in SUBSYSTEM_DIR.items():
        sub_path = DIG_TWIN / folder
        if not sub_path.exists(): continue
        for exp_dir in sorted(sub_path.iterdir()):
            if not exp_dir.is_dir() or exp_dir.name.startswith('.'): continue
            for run_dir in sorted(exp_dir.iterdir()):
                if not run_dir.is_dir() or run_dir.name.startswith('.'): continue
                csv_files = list(run_dir.glob('faultDataset_*.csv'))
                cause_files = list(run_dir.glob('causes.json'))
                if not csv_files or not cause_files: continue
                try:
                    causes = json.loads(cause_files[0].read_text())
                    cause_start = causes.get('cause_start_at', 0.0)
                    wide = load_wide(csv_files[0], ALL_FEATURES)
                    normal = wide[wide.index < cause_start]
                    if len(normal) < WINDOW + 1:
                        continue
                    arr = normal.values.astype(np.float32)
                    arr = np.clip((arr - col_min) / col_range, 0.0, 1.0)
                    arr = arr[:, indices]
                    wins.append(make_windows(arr))
                except Exception:
                    pass

    dig_twin_wins = sum(len(w) for w in wins)
    print(f'    dig_twin 윈도우: {dig_twin_wins}개')

    # (B) real_op windows (전체가 정상)
    real_op_wins = 0
    if REAL_OP.exists():
        real_csvs = sorted(REAL_OP.glob('*.csv'))
        for csv_f in real_csvs:
            try:
                wide = load_wide(csv_f, ALL_FEATURES)
                if len(wide) < WINDOW + 1:
                    continue
                arr = wide.values.astype(np.float32)
                arr = np.clip((arr - col_min) / col_range, 0.0, 1.0)
                arr = arr[:, indices]
                w = make_windows(arr)
                wins.append(w)
                real_op_wins += len(w)
            except Exception:
                pass
    print(f'    real_op 윈도우: {real_op_wins}개')

    if not wins:
        print('    데이터 없음 — 스킵')
        continue

    X = torch.tensor(np.concatenate(wins))
    loader = DataLoader(TensorDataset(X), batch_size=BATCH_SIZE, shuffle=True)
    print(f'    총 학습 윈도우: {len(X)}개')

    model = LSTMAutoencoder(n_feat).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)

    model.train()
    for ep in range(1, EPOCHS + 1):
        total = 0.0
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            opt.zero_grad()
            loss = ((model(xb) - xb) ** 2).mean()
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
        sch.step()
        if ep % 10 == 0 or ep == 1:
            print(f'    epoch {ep:3d}/{EPOCHS}  mse={total/len(X):.6f}')

    # Threshold 계산 (전체 학습 데이터 기준)
    model.eval()
    errors = []
    with torch.no_grad():
        for (xb,) in DataLoader(TensorDataset(X), batch_size=256):
            xb = xb.to(DEVICE)
            mse = ((model(xb) - xb) ** 2).mean(dim=(1, 2)).cpu().numpy()
            errors.extend(mse.tolist())
    thr = float(np.mean(errors) + 3 * np.std(errors))

    # 저장
    torch.save(model.state_dict(), LSTM_DIR / f'{subsystem}_lstm.pt')
    info = {
        'n_features': n_feat,
        'hidden_size': HIDDEN,
        'latent_dim': LATENT,
        'n_layers': N_LAYERS,
        'window_size': WINDOW,
        'threshold_3sigma': thr,
        'train_windows': int(len(X)),
        'train_error_mean': float(np.mean(errors)),
        'train_error_std': float(np.std(errors)),
    }
    with open(LSTM_DIR / f'{subsystem}_lstm_info.json', 'w') as f:
        json.dump(info, f, indent=2)
    print(f'    → threshold_3sigma={thr:.6f}  (mean={np.mean(errors):.6f}, std={np.std(errors):.6f})')

print('\n학습 완료!')

In [ ]:
# ── [E-4] 압축 + 다운로드 ──
print('[4/4] 압축...')
zip_path = OUT_DIR / 'edge_agent_data.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in PROC_DIR.iterdir():
        zf.write(f, f'processed_data_v2/{f.name}')
    for f in LSTM_DIR.iterdir():
        zf.write(f, f'models_lstm/{f.name}')

print(f'\n완료! 출력 파일:')
with zipfile.ZipFile(zip_path) as zf:
    for name in sorted(zf.namelist()):
        print(f'  {name}  ({zf.getinfo(name).file_size/1024:.1f} KB)')
print(f'\n→ {zip_path}')

# Colab 다운로드
from google.colab import files
files.download(str(zip_path))
